# Forecast Error Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

print('Loading data...')
try:
    actuals = requests.get('http/api/actuals').json()
    forecasts = requests.get('http://localhost:3000/api/forecasts').json()
    actuals_df = pd.DataFrame(actuals)
    forecasts_df = pd.DataFrame(forecasts)
    print(f'✓ Loaded {len(actuals_df)} actual and {len(forecasts_df)} forecast records')
except Exception as e:
    print(f'Error: {e}')
    actuals_df = pd.DataFrame()
    forecasts_df = pd.DataFrame()

In [ ]:
if not actuals_df.empty and not forecasts_df.empty:
    actuals_df['startTime'] = pd.to_datetime(actuals_df['startTime'])
    forecasts_df['startTime'] = pd.to_datetime(forecasts_df['startTime'])
    forecasts_df['publishTime'] = pd.to_datetime(forecasts_df['publishTime'])
    
    merged = forecasts_df.merge(actuals_df, on='startTime', suffixes=('_forecast', '_actual'))
    merged['error'] = merged['generation_actual'] - merged['generation_forecast']
    merged['abs_error'] = abs(merged['error'])
    
    print(f'Merged {len(merged)} forecast-actual pairs')
else:
    merged = pd.DataFrame()
    print('No data')

## Error Statistics

In [ ]:
if not merged.empty:
    print('Error Statistics (MW):')
    print(f'  Mean:   {merged["abs_error"].mean():.2f}')
    print(f'  Median: {merged["abs_error"].median():.2f}')
    print(f'  P99:    {merged["abs_error"].quantile(0.99):.2f}')

## Error by Forecast Horizon

In [ ]:
if not merged.empty and 'publishTime' in merged.columns:
    merged['horizon_h'] = (merged['startTime'] - merged['publishTime']).dt.total_seconds() / 3600
    
    horizon_stats = merged.groupby(merged['horizon_h'].astype(int))['abs_error'].agg([
        'mean', 'median', 
        ('p99', lambda x: x.quantile(0.99))
    ]).reset_index()
    
    print('Error by Forecast Horizon (hours):')
    print(horizon_stats.to_string(index=False))
    
    if len(horizon_stats) > 1:
        plt.figure(figsize=(10, 4))
        plt.plot(horizon_stats['horizon_h'], horizon_stats['mean'], marker='o', label='Mean')
        plt.plot(horizon_stats['horizon_h'], horizon_stats['median'], marker='s', label='Median')
        plt.plot(horizon_stats['horizon_h'], horizon_stats['p99'], marker='^', label='P99')
        plt.xlabel('Forecast Horizon (hours)')
        plt.ylabel('Error (MW)')
        plt.title('Error by Forecast Horizon')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

## Error by Time of Day

In [ ]:
if not merged.empty:
    merged['hour'] = merged['startTime'].dt.hour
    merged['period'] = pd.cut(merged['hour'], bins=[0, 6, 12, 18, 24], 
                              labels=['Night (0-6)', 'Morning (6-12)', 'Afternoon (12-18)', 'Evening (18-24)'], 
                              include_lowest=True)
    
    time_stats = merged.groupby('period')['abs_error'].agg([
        'mean', 'median', 
        ('p99', lambda x: x.quantile(0.99)),
        'count'
    ]).reset_index()
    
    print('Error by Time of Day:')
    print(time_stats.to_string(index=False))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    
    axs[0].bar(range(len(time_stats)), time_stats['mean'])
    axs[0].set_xticks(range(len(time_stats)))
    axs[0].set_xticklabels(time_stats['period'], rotation=45)
    axs[0].set_ylabel('Mean Error (MW)')
    axs[0].set_title('Mean Error by Time of Day')
    axs[0].grid(True, alpha=0.3, axis='y')
    
    axs[1].bar(range(len(time_stats)), time_stats['p99'])
    axs[1].set_xticks(range(len(time_stats)))
    axs[1].set_xticklabels(time_stats['period'], rotation=45)
    axs[1].set_ylabel('P99 Error (MW)')
    axs[1].set_title('P99 Error by Time of Day')
    axs[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()